# Component I: Product Review Generator (E-Commerce)

# (Install & Import)

In [1]:
!pip install transformers datasets accelerate -q

import torch, math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)

## Load Pre-trained GPT-2 Model

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')
model = GPT2LMHeadModel.from_pretrained('distilgpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

## Generate Baseline Reviews (Before Fine-Tuning)

In [ ]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("=== BASELINE REVIEWS ===\n")
baseline = {}

for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"Prompt: {p}\nOutput: {baseline[p]}\n")

## Prepare Dataset for Fine-Tuning

In [ ]:
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "the tablet screen is bright and colorful which makes watching movies a great experience.",
    "i love this fitness tracker because it motivates me to reach my daily exercise goals.",
    "this bluetooth speaker is compact but delivers surprisingly loud and clear audio.",
    "the delivery was fast and the product was packed securely with no damage at all.",
    "excellent value for money and the build quality feels premium despite the affordable price.",
    "the customer service team was very helpful when i had questions about the product features.",
    "this camera takes stunning photos in low light and the video recording quality is very smooth.",
    "i have been using this product for three months and it still works perfectly like day one.",
    "the design is sleek and modern and it looks great on my desk next to my other gadgets.",
    "easy to set up right out of the box and the instructions were clear and simple to follow.",
    "highly recommend this product to anyone looking for quality and reliability at a fair price.",
    "the software updates keep adding new features which makes this purchase even more worthwhile.",
    "best purchase i made this year and i would definitely buy from this brand again."
]

dataset = Dataset.from_dict({"text": corpus})

tokenized = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=128, padding="max_length"),
    batched=True,
    remove_columns=["text"]
)

split = tokenized.train_test_split(test_size=0.15, seed=42)

## Fine-Tune GPT-2 Model

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator
)

trainer.train()

## Generate Reviews After Fine-Tuning

In [ ]:
eval_res = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_res['eval_loss']):.2f}\n")

print("=== FINE-TUNED REVIEWS ===\n")

for p in review_prompts:
    ft = generate_text(model, tokenizer, p)
    print(f"Prompt: {p}")
    print(f"Baseline:   {baseline[p][:120]}")
    print(f"Fine-Tuned: {ft[:120]}\n")

# Component II: Recipe Generator (Food-Tech)

# Fine-Tuning GPT-2 for Recipe Generation

In [ ]:
tokenizer2 = GPT2Tokenizer.from_pretrained('distilgpt2')
model2 = GPT2LMHeadModel.from_pretrained('distilgpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

## Generate Baseline Recipes

In [ ]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare a chocolate cake"
]

baseline2 = {}

print("=== BASELINE RECIPES ===\n")

for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f"Prompt: {p}\nOutput: {baseline2[p]}\n")

## Prepare Recipe Dataset

In [ ]:
recipes = [
    "to make butter chicken start by marinating chicken pieces in yogurt with spices.",
    "heat butter and fry onions until golden then add ginger garlic paste.",
    "add tomato puree and cook until oil separates.",
    "add chicken and cook until done.",
    "finish with cream and serve hot.",

    "for pasta carbonara boil pasta until al dente.",
    "fry pancetta until crispy.",
    "mix eggs cheese and pepper.",
    "combine pasta with pancetta and egg mixture.",
    "serve immediately with parmesan.",

    "to prepare stir fry heat oil in wok.",
    "add vegetables and stir for few minutes.",
    "add sauces and cook well.",
    "serve with rice."
]

dataset2 = Dataset.from_dict({"text": recipes})

tok2 = dataset2.map(
    lambda x: tokenizer2(x["text"], truncation=True, max_length=128, padding="max_length"),
    batched=True,
    remove_columns=["text"]
)

split2 = tok2.train_test_split(test_size=0.15, seed=42)

## Fine-Tune Recipe Model

In [ ]:
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir="./gpt2-recipes",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=split2["train"],
    eval_dataset=split2["test"],
    data_collator=collator2
)

trainer2.train()

## Generate Recipes After Fine-Tuning

In [ ]:
eval2 = trainer2.evaluate()
print(f"Perplexity: {math.exp(eval2['eval_loss']):.2f}\n")

print("=== FINE-TUNED RECIPES ===\n")

for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)
    print(f"Prompt: {p}")
    print(f"Baseline:   {baseline2[p][:120]}")
    print(f"Fine-Tuned: {ft[:120]}\n")

## Save Product Review Results

In [ ]:
import pandas as pd

results_reviews = []

for p in review_prompts:
    ft = generate_text(model, tokenizer, p)

    results_reviews.append({
        "Prompt": p,
        "Baseline": baseline[p],
        "Fine_Tuned": ft
    })

df_reviews = pd.DataFrame(results_reviews)

# Save as CSV
df_reviews.to_csv("product_review_results.csv", index=False)

print("Saved product review results!")
df_reviews

## Save Recipe Generation Results

In [ ]:
results_recipes = []

for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)

    results_recipes.append({
        "Prompt": p,
        "Baseline": baseline2[p],
        "Fine_Tuned": ft
    })

df_recipes = pd.DataFrame(results_recipes)

df_recipes.to_csv("recipe_results.csv", index=False)

print("Saved recipe results!")
df_recipes